# RUPA-DSA — train UCF-Crime / XD-Violence on Kaggle

# Train UCF

In [ ]:
DATASET = "ucf"
MAX_EPOCHS = 10           
UCF_BATCH_SIZE = 16      
NUM_WORKERS = 2
SEED = 234

RESUME = False
WARM_START = False 

REPO_URL = "https://github.com/Sharvuz/RUPA-DSA-v2.git"
REPO_REF = None

# TRAIN XD

In [1]:
# ========================= USER CONFIG =========================
DATASET = "xd"             
MAX_EPOCHS = 10            
XD_BATCH_SIZE = 16       
NUM_WORKERS = 2
SEED = 234

RESUME = False
WARM_START = False 

REPO_URL = "https://github.com/Sharvuz/RUPA-DSA-v2.git"
REPO_REF = None

## cài pytorch p100

In [ ]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-cache-dir",
    "--force-reinstall",
    "torch==2.6.0",
    "torchvision==0.21.0",
    "torchaudio==2.6.0",
    "--index-url", "https://download.pytorch.org/whl/cu124"
], check=True)

print("Đã cài PyTorch tương thích P100. Hãy Restart Session.")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Architectures:", torch.cuda.get_arch_list())

assert torch.cuda.is_available()
assert "sm_60" in torch.cuda.get_arch_list()

x = torch.randn(1024, 1024, device="cuda")
y = x @ x
print("P100 CUDA test thành công:", y.device)

## 1. Clone RUPA-DSA and prepare the GPU environment

Kaggle clones the source directly from `Sharvuz/RUPA-DSA`. No source code or base64 overlay is embedded in this notebook. Set `REPO_REF` in the config cell when you want to pin an exact commit or tag.

In [2]:
import json, os, random, shutil, subprocess, sys, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "RUPA-DSA"
ARTIFACTS = WORK / "rupa_artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    shutil.rmtree(REPO)
clone_cmd = ["git", "clone", "--depth", "1"]
if REPO_REF:
    clone_cmd += ["--branch", REPO_REF]
clone_cmd += [REPO_URL, str(REPO)]
subprocess.run(clone_cmd, check=True)
os.chdir(REPO)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ftfy", "regex", "einops==0.8.0", "ipdb", "scikit-learn", "pandas"
], check=True)

import numpy as np
import pandas as pd
import torch

print("Repository:", REPO_URL, "ref:", REPO_REF or "default branch")
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Enable Accelerator = GPU in Kaggle Notebook Settings."
assert "RUPA-DSA" in (REPO / "src/model.py").read_text(encoding="utf-8")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("Direct GitHub clone ready.")

Cloning into '/kaggle/working/RUPA-DSA'...


Repository: https://github.com/Sharvuz/RUPA-DSA-v2.git ref: default branch
PyTorch: 2.6.0+cu124
CUDA runtime: 12.4
GPU: Tesla P100-PCIE-16GB
Direct GitHub clone ready.


## 2. Find/extract features and rebuild CSV paths

No Kaggle Dataset slug or mount name is hard-coded. ZIP extraction is only used when the required `.npy` files are not already mounted.

# tao csv UCF

# cả 2 dataset đều dc đổi tên hàng loạt file từ "#" thành "+". Vì kaggle hạn chế kí tự "#"

In [ ]:
from collections import defaultdict
from pathlib import PurePosixPath

if DATASET == "ucf":
    csv_specs = [
        ("list/ucf_CLIP_rgb.csv", "ucf_train.csv"),
        ("list/ucf_CLIP_rgbtest.csv", "ucf_test.csv"),
    ]u
    zip_hints = ["ucfclipfeatures"]
else:
    csv_specs = [
        ("list/xd_CLIP_rgb.csv", "xd_train.csv"),
        ("list/xd_CLIP_rgbtest.csv", "xd_test.csv"),
    ]
    zip_hints = ["xdtrainclipfeatures", "xdtestclipfeatures"]

def basename(path_string):
    return PurePosixPath(str(path_string).replace("\\", "/")).name

required_names = set()
for csv_rel, _ in csv_specs:
    frame = pd.read_csv(REPO / csv_rel)
    required_names.update(basename(p) for p in frame["path"])

feature_roots = [INPUT]

def indexed_names(roots):
    return {p.name for root in roots for p in root.rglob("*.npy")}

missing_before = required_names - indexed_names(feature_roots)
if missing_before:
    extract_root = WORK / f"rupa_features_{DATASET}"
    extract_root.mkdir(parents=True, exist_ok=True)
    archives = [
        p for p in INPUT.rglob("*.zip")
        if any(hint in p.stem.lower() for hint in zip_hints)
    ]
    if archives:
        print(f"Extracting {len(archives)} feature archive(s); this can take several minutes...")
    for archive_path in archives:
        destination = (extract_root / archive_path.stem).resolve()
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                target = (destination / member.filename).resolve()
                if destination not in target.parents and target != destination:
                    raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
            archive.extractall(destination)
        print("Extracted:", archive_path.name)
    feature_roots.append(extract_root)

index = defaultdict(list)
for root in feature_roots:
    for path in root.rglob("*.npy"):
        if path.name in required_names:
            index[path.name].append(path)

def rank_candidate(candidate, original):
    original_parts = [x.lower() for x in str(original).replace("\\", "/").split("/")[-4:]]
    candidate_text = str(candidate).lower()
    return sum(part in candidate_text for part in original_parts)

def rewrite_csv(csv_rel, output_name):
    frame = pd.read_csv(REPO / csv_rel)
    resolved, missing, ambiguous = [], [], 0
    for original in frame["path"].astype(str):
        matches = index.get(basename(original), [])
        if not matches:
            missing.append(basename(original))
            resolved.append("")
            continue
        matches = sorted(matches, key=lambda p: rank_candidate(p, original), reverse=True)
        resolved.append(str(matches[0]))
        ambiguous += int(len(matches) > 1)
    if missing:
        raise FileNotFoundError(
            f"{csv_rel}: missing {len(missing)}/{len(frame)} features; examples={missing[:10]}"
        )
    output = ARTIFACTS / output_name
    frame["path"] = resolved
    frame.to_csv(output, index=False)
    print(f"{output_name}: rows={len(frame)}, missing=0, duplicate basenames={ambiguous}")
    return output

train_csv = rewrite_csv(*csv_specs[0])
test_csv = rewrite_csv(*csv_specs[1])

sample_path = Path(pd.read_csv(train_csv).iloc[0]["path"])
sample = np.load(sample_path, mmap_mode="r")
assert sample.ndim == 2 and sample.shape[-1] == 512, (
    f"Expected CLIP feature [T,512], got {sample.shape} from {sample_path}"
)
print("Feature preflight:", sample_path.name, sample.shape)

# tao CSV XD

In [3]:
import os
import shutil
from pathlib import Path

print("Đang quét toàn bộ hệ thống Kaggle để tìm file .npy...")

# Quét sạch sẽ toàn bộ /kaggle/input và /kaggle/working
all_npy_paths = list(Path("/kaggle/input").rglob("*.npy")) + list(Path("/kaggle/working").rglob("*.npy"))

if not all_npy_paths:
    print("CẢNH BÁO ĐỎ: KHÔNG TÌM THẤY FILE .NPY NÀO! Bạn đã Add Data XD-Violence vào Kaggle chưa?")
else:
    print(f"Radar đã quét thấy {len(all_npy_paths)} file .npy trên hệ thống!")

# Tạo từ điển bản đồ: Tên file -> Đường dẫn tuyệt đối
file_map = {path.name: path for path in all_npy_paths}

def rewrite_csv(csv_rel, output_name):
    input_csv = REPO / csv_rel
    output = ARTIFACTS / output_name
    print(f"Đang đồng bộ danh sách {input_csv.name}...")
    
    missing = []
    is_first_line = True
    
    with input_csv.open("r", encoding="utf-8") as f, output.open("w", encoding="utf-8") as out:
        for line in f:
            line = line.strip()
            if not line: continue
            
            parts = line.split(",")
            feature_path = parts[0]
            
            if is_first_line and ("path" in feature_path.lower() or "name" in feature_path.lower()):
                out.write(line + "\n")
                is_first_line = False
                continue
            is_first_line = False
            
            feature_basename = Path(feature_path).name
            
            # Thử mọi cách "biến dạng" do Kaggle tự ý đổi tên file
            possible_names = [
                feature_basename,                                # Tên gốc chuẩn
                feature_basename.replace("#", "+"),              # Kaggle đổi # thành +
                feature_basename.replace("#", ""),               # Kaggle ăn mất dấu #
                feature_basename.replace("#", "_"),              # Kaggle đổi # thành _
            ]
            
            match_found = False
            for p_name in possible_names:
                if p_name in file_map:
                    parts[0] = str(file_map[p_name])
                    out.write(",".join(parts) + "\n")
                    match_found = True
                    break
                    
            if not match_found:
                missing.append(feature_basename)
                
    if missing:
        raise FileNotFoundError(f"Lỗi: Vẫn thiếu {len(missing)} file. Ví dụ: {missing[:3]}")
        
    return output

# Tiến hành khớp nối
train_csv = rewrite_csv(f"list/{DATASET}_CLIP_rgb.csv", f"{DATASET}_train.csv")
test_csv = rewrite_csv(f"list/{DATASET}_CLIP_rgbtest.csv", f"{DATASET}_test.csv")

print("✅ Đã lập bản đồ và đồng bộ dữ liệu XD-Violence thành công 100%!")

Đang quét toàn bộ hệ thống Kaggle để tìm file .npy...
Radar đã quét thấy 67046 file .npy trên hệ thống!
Đang đồng bộ danh sách xd_CLIP_rgb.csv...
Đang đồng bộ danh sách xd_CLIP_rgbtest.csv...
✅ Đã lập bản đồ và đồng bộ dữ liệu XD-Violence thành công 100%!


# Train UCF

In [5]:
import json, os, subprocess, sys, time
from pathlib import Path

dataset_dir = ARTIFACTS / DATASET
dataset_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = dataset_dir / f"checkpoint_{DATASET}.pth"
model_path = dataset_dir / f"best_{DATASET}.pth"
train_csv = ARTIFACTS / f"{DATASET}_train.csv"
test_csv = ARTIFACTS / f"{DATASET}_test.csv"

cmd = [
    sys.executable, f"src/{DATASET}_train.py",
    "--train-list", str(train_csv),
    "--test-list", str(test_csv),
    "--model-path", str(model_path),
    "--checkpoint-path", str(checkpoint_path),
    
    "--max-epoch", str(MAX_EPOCHS),
    "--batch-size", str(UCF_BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED),
    
    # Bật tính năng RUPA và Otsu
    "--rupa-use", "true",

    # Các trọng số mặc định chuẩn xác của bản v2
    "--routing-det-weight", "0.5",
    "--routing-rec-weight", "0.3",
    "--routing-sem-weight", "0.2",
    "--loss-residual-weight", "1.0",
    "--loss-reconstructed-normal-weight", "1.0",
    "--loss-dnp-normal-weight", "0.1",
    "--loss-consistency-weight", "1.0",
    "--loss-gather-weight", "1.0",
]

if RESUME:
    cmd.extend(["--use-checkpoint", "true"])

log_path = dataset_dir / f"train_{DATASET}_v2_3831.log"
started = time.time()
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

print(f"Bắt đầu train UCF-Crime bản v2 với {MAX_EPOCHS} Epochs...")

with log_path.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd, cwd=REPO, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")

print(f"Hoàn thành xuất sắc trong {(time.time() - started)/3600:.2f} giờ!")

NameError: name 'UCF_BATCH_SIZE' is not defined

# TRAIN XD

In [4]:
## import json, os, subprocess, sys, time
from pathlib import Path

# Đảm bảo ARTIFACTS và REPO đã được định nghĩa ở các cell chuẩn bị trước đó
dataset_dir = ARTIFACTS / DATASET
dataset_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = dataset_dir / f"checkpoint_{DATASET}.pth"
model_path = dataset_dir / f"best_{DATASET}.pth"
train_csv = ARTIFACTS / f"{DATASET}_train.csv"
test_csv = ARTIFACTS / f"{DATASET}_test.csv"

cmd = [
    sys.executable, "-u", f"src/{DATASET}_train.py",  # Thêm flag -u (unbuffered) cho Python
    "--train-list", str(train_csv),
    "--test-list", str(test_csv),
    "--model-path", str(model_path),
    "--checkpoint-path", str(checkpoint_path),
    
    "--max-epoch", str(MAX_EPOCHS),
    "--batch-size", str(XD_BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED),

    "--lr", "5e-6",

    # ------------------ CẤU HÌNH RUPA v2 ------------------
    "--rupa-use", "true",
    "--adaptive_normal_selection", "false",   # Tắt Otsu để tránh nhiễu trên XD-Violence

    # Trọng số định tuyến (Routing)
    "--routing-det-weight", "0.5",  
    "--routing-rec-weight", "0.3",   
    "--routing-sem-weight", "0.2",   
    
    # Trọng số Loss 
    "--loss-residual-weight", "1.0",
    "--loss-reconstructed-normal-weight", "1.0",
    "--loss-dnp-normal-weight", "0.1",
    "--loss-consistency-weight", "1.0",
    "--loss-gather-weight", "1.0",
]

if RESUME:
    cmd.extend(["--use-checkpoint", "true"])

log_path = dataset_dir / f"train_{DATASET}_v2.log"
started = time.time()

# Ép hệ điều hành tắt buffer
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

print(f"🚀 Bắt đầu train XD-Violence bản v2 (Train từ con số 0) với {MAX_EPOCHS} Epochs...\n", flush=True)

with log_path.open("w", encoding="utf-8") as log_file:
    # bufsize=1 và text=True giúp đọc log theo từng dòng real-time
    process = subprocess.Popen(
        cmd, cwd=REPO, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    
    # Vòng lặp đọc log ngay khi có dòng mới
    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()       
        log_file.write(line)
        log_file.flush()
        
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f"❌ Training thất bại với exit code {return_code}")

print(f"\n✅ Hoàn thành xuất sắc trong {(time.time() - started)/3600:.2f} giờ!")

🚀 Bắt đầu train XD-Violence bản v2 (Train từ con số 0) với 10 Epochs...

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
epoch: 1 | step: 4800 | loss1: 0.4503 | loss2: 1.4172 | loss3: 0.7700 | loss4: 1.9161 | loss5: 0.8053 | consistency_loss: 0.0183 | g_loss: 0.4205 | dnp_normal_loss: 0.9668
epoch: 1 | step: 9600 | loss1: 0.3817 | loss2: 1.2219 | loss3: 0.4612 | loss4: 1.7493 | loss5: 0.7253 | consistency_loss: 0.0147 | g_loss: 0.1481 | dnp_normal_loss: 0.9338
epoch: 1 | step: 14400 | loss1: 0.3513 | loss2: 1.0902 | loss3: 0.2669 | loss4: 1.6103 | loss5: 0.6985 | consistency_loss: 0.0099 | g_loss: 0.2168 | dnp_normal_loss: 0.9025
epoch: 1 | step: 19200 | loss1: 0.3309 | loss2: 0.9863 | loss3: 0.1391 | loss4: 1.4924 | loss5: 0.6852 | consistency_loss: 0.0104 | g_loss: 0.2096 | dnp_normal_loss: 0.8456
epoch: 1 | step: 24000 | loss1: 0.314